# Topological Data Analysis — CFM ITS1 Fungi

Persistent homology on the Cacao Flower Microbiome (CFM) ITS1 fungal community
in Aitchison (CLR-PCA) space.

**Dataset:** CFM ITS1 OTU97 fungi — biosamples only (decontamination performed in R decontam and controls removed).  
**Groups:** `bagged_flower` (pollinator-excluded) vs `unbagged_flower` (open).  
**Metric:** Aitchison distance (CLR-Euclidean) — valid metric, satisfies triangle inequality.

**Workflow:**
1. Load feature table + metadata
2. CLR transform (multiplicative zero replacement → CLR)
3. PCA (Aitchison PCA = PCA on CLR matrix) + scree plot
4. Persistent homology — all samples (GUDHI Alpha complex)
5. PH per group — bagged vs unbagged
6. Topological comparison (Wasserstein distance + permutation test)
7. Mapper

## 1. Imports

In [ ]:
from pathlib import Path

import biom
import gudhi
import gudhi.wasserstein
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.decomposition import PCA
from skbio.stats.composition import clr, multi_replace

print(f"GUDHI version: {gudhi.__version__}")

## 2. Configuration

In [ ]:
ROOT        = Path("..").resolve()   # notebooks/../ = project root
BIOM_PATH   = ROOT / "results" / "biom" / "cfm_its1_otu97_fungi_biosamples_feature-table.biom"
META_PATH   = ROOT / "data" / "sample_metadata_with_microscopy.txt"
FIGURES_DIR = ROOT / "results" / "figures" / "tda"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED      = 42
N_PCA_COMPONENTS = 150   # compute 100, choose elbow from scree plot
N_PERMUTATIONS   = 999   # permutation test iterations

# OTUs present in fewer than this fraction of samples are dropped before CLR.
# 0.10 = must appear in at least 10% of samples (~29 out of 294).
# Increase to 0.20 for a more conservative core community.
PREVALENCE_THRESHOLD = 0.10

# Colors for bagged vs unbagged
GROUP_COLORS = {
    "bagged_flower":   "#4C72B0",
    "unbagged_flower": "#DD8452",
}

# Multipage PDF collector: append all figures as they are created, close in the last cell
PDF_ALL = PdfPages(FIGURES_DIR / "07b_cfm_its1_tda_all_figures.pdf")

def save_fig(fig, name):
    """Save as individual PDF, PNG at 300 dpi, and append to multipage PDF."""
    fig.savefig(FIGURES_DIR / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{name}.png", bbox_inches="tight", dpi=300)
    PDF_ALL.savefig(fig, bbox_inches="tight")

## 3. Load data

In [ ]:
print("Loading feature table ...")
table = biom.load_table(str(BIOM_PATH))
ft = table.to_dataframe(dense=True).T   # samples x OTUs
print(f"  {ft.shape[0]:,} samples x {ft.shape[1]:,} OTUs")

print("Loading metadata ...")
meta = pd.read_csv(META_PATH, sep="\t", index_col="sample-id", skiprows=[1])

# Align samples present in both
common = ft.index.intersection(meta.index)
ft, meta = ft.loc[common], meta.loc[common]

# Keep only bagged / unbagged flowers (drop QC controls if any remain)
mask = meta["sample_type"].isin(["bagged_flower", "unbagged_flower"])
ft   = ft[mask].copy()
meta = meta[mask].copy()

group_labels = meta["sample_type"].values

print(f"After filtering: {len(ft):,} samples x {ft.shape[1]:,} OTUs")
print(meta["sample_type"].value_counts().to_string())

## 4. Prevalence filtering

ITS1 OTU tables are very sparse (median ~97% zeros per sample). With that many zeros,
`multi_replace` would impute the same pseudocount to almost all positions — the CLR
coordinates would reflect imputation noise rather than real composition.

Standard CoDA practice: **keep only OTUs that appear in at least X% of samples**
(the "core" community), then apply CLR on this filtered table. Rare OTUs dropped here
are real, but their presence/absence pattern is too sparse to contribute reliable
continuous-distance information — they are better analysed via binary (Jaccard) methods.

The curve below shows how many OTUs survive at each prevalence threshold.

In [ ]:
ft_arr = ft.values.astype(float)

# OTU prevalence = fraction of samples where count > 0
otu_prevalence = (ft_arr > 0).mean(axis=0)

thresholds = np.linspace(0, 0.5, 200)
n_otus_retained = [(otu_prevalence >= t).sum() for t in thresholds]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # left: retention curve, right: prevalence histogram

# Left: OTUs retained vs threshold
ax = axes[0]
ax.plot(thresholds * 100, n_otus_retained, color="#2c7bb6", linewidth=1.8)
ax.axvline(
    PREVALENCE_THRESHOLD * 100,
    color="#d7191c", linestyle="--", linewidth=1.5,
    label=f"threshold = {PREVALENCE_THRESHOLD:.0%}\n"
          f"({(otu_prevalence >= PREVALENCE_THRESHOLD).sum()} OTUs retained)",
)
ax.set_xlabel("Prevalence threshold (%)", fontsize=10)
ax.set_ylabel("OTUs retained", fontsize=10)
ax.set_title("OTUs retained vs prevalence threshold", fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

# Right: OTU prevalence histogram (log y: most OTUs are very rare)
ax2 = axes[1]
ax2.hist(
    otu_prevalence * 100,
    bins=50,
    color="#abd9e9", edgecolor="#2c7bb6", linewidth=0.4,
    log=True,   # log y-axis makes the rare-OTU peak and persistent-OTU tail both visible
)
ax2.axvline(
    PREVALENCE_THRESHOLD * 100,
    color="#d7191c", linestyle="--", linewidth=1.5,
    label=f"threshold = {PREVALENCE_THRESHOLD:.0%}",
)
ax2.set_xlabel("OTU prevalence (%)", fontsize=10)
ax2.set_ylabel("Number of OTUs (log scale)", fontsize=10)
ax2.set_title("Prevalence distribution — all OTUs", fontsize=11)
ax2.legend(fontsize=9)

plt.tight_layout()
save_fig(fig, "07b_cfm_its1_tda_prevalence")
plt.show()

In [ ]:
# Apply prevalence filter
keep = otu_prevalence >= PREVALENCE_THRESHOLD
ft_filtered  = ft.loc[:, keep].copy()
ft_filt_arr  = ft_filtered.values.astype(float)

zero_frac_after = (ft_filt_arr == 0).mean(axis=1)

print(f"OTUs before filtering: {ft_arr.shape[1]:,}")
print(f"OTUs after  filtering: {ft_filt_arr.shape[1]:,}")
print(f"Zero fraction before:  median={(ft_arr == 0).mean(axis=1).mean():.2%}")
print(f"Zero fraction after:   median={np.median(zero_frac_after):.2%}, max={zero_frac_after.max():.2%}")

In [ ]:
# ---- Sparsity profile: zeros, singletons, doubletons per sample ----

def sparsity_barplot(ax, counts, title, normalize="all"):
    """
    Stacked barplot of per-sample feature sparsity.
    X: samples ordered by group, no tick labels, no gaps (width=1.0).
    Stacks and denominator depend on normalize:
      "all"     : denominator = total features; stacks = zero / singleton / doubleton / >=3
      "nonzero" : denominator = detected (non-zero) features per sample; stacks = singleton / doubleton / >=3
    """
    x  = np.arange(len(counts))
    kw = dict(width=1.0, edgecolor="none")  # width=1.0 removes gaps between bars

    if normalize == "all":
        n_feat   = counts.shape[1]           # total features (same for all samples)
        pct_zero = (counts == 0).sum(axis=1) / n_feat * 100
        pct_sing = (counts == 1).sum(axis=1) / n_feat * 100
        pct_doub = (counts == 2).sum(axis=1) / n_feat * 100
        pct_rest = 100 - pct_zero - pct_sing - pct_doub
        ax.bar(x, pct_zero,                                     color="#d9d9d9", label="zero",           **kw)
        ax.bar(x, pct_sing, bottom=pct_zero,                    color="#fc8d59", label="singleton (=1)", **kw)
        ax.bar(x, pct_doub, bottom=pct_zero + pct_sing,         color="#91bfdb", label="doubleton (=2)", **kw)
        ax.bar(x, pct_rest, bottom=pct_zero + pct_sing + pct_doub, color="#2166ac", label=">=3",        **kw)
        ax.set_ylabel("Features (%)")
    else:  # normalize == "nonzero"
        # Per-sample denominator: only detected (non-zero) features; NaN for empty samples
        n_detected = (counts > 0).sum(axis=1).replace(0, np.nan)
        pct_sing = (counts == 1).sum(axis=1) / n_detected * 100
        pct_doub = (counts == 2).sum(axis=1) / n_detected * 100
        pct_rest = 100 - pct_sing - pct_doub
        ax.bar(x, pct_sing,                           color="#fc8d59", label="singleton (=1)", **kw)
        ax.bar(x, pct_doub, bottom=pct_sing,          color="#91bfdb", label="doubleton (=2)", **kw)
        ax.bar(x, pct_rest, bottom=pct_sing + pct_doub, color="#2166ac", label=">=3",          **kw)
        ax.set_ylabel("Detected features (%)")

    ax.set_xlim(0, len(counts))
    ax.set_ylim(0, 100)
    ax.set_xticks([])
    ax.set_xlabel(f"Samples (n={len(counts)})")
    ax.set_title(title)


# Sort samples by group for consistent visual ordering across all panels
order = meta.sort_values("sample_type").index.intersection(ft.index)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))  # rows: all features / detected only; cols: before / after filter
fig.suptitle("ITS1 OTU97 — feature sparsity per sample (sorted by sample_type)", fontsize=12)

# Row 1: denominator = all features — zeros dominate, singletons/doubletons are invisible slivers
sparsity_barplot(axes[0, 0], ft.loc[order],          f"Before filter — all features ({ft.shape[1]} OTUs)",          normalize="all")
sparsity_barplot(axes[0, 1], ft_filtered.loc[order], f"After filter — all features ({ft_filtered.shape[1]} OTUs)",  normalize="all")

# Row 2: denominator = detected (non-zero) features only — singletons/doubletons now clearly visible
sparsity_barplot(axes[1, 0], ft.loc[order],          f"Before filter — detected only ({ft.shape[1]} OTUs)",         normalize="nonzero")
sparsity_barplot(axes[1, 1], ft_filtered.loc[order], f"After filter — detected only ({ft_filtered.shape[1]} OTUs)", normalize="nonzero")

# Single legend at bottom centre: Row 1 handles cover all 4 stacks; Row 2 is a subset with the same colours
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.04))

fig.tight_layout()
save_fig(fig, "07b_cfm_its1_tda_sparsity")
plt.show()

## 5. CLR transform

CLR on the prevalence-filtered table. After filtering, sparsity is much lower and
multiplicative replacement introduces less noise. Euclidean distances in CLR space
equal Aitchison distances — the correct metric for compositional data in TDA.

In [ ]:
# Normalize to relative abundances (required by multi_replace)
row_sums    = ft_filt_arr.sum(axis=1, keepdims=True)
ft_rel      = ft_filt_arr / row_sums

print("Applying multiplicative zero replacement ...")
ft_replaced = multi_replace(ft_rel)

print("Applying CLR transform ...")
X_clr = clr(ft_replaced)   # shape: (n_samples, n_otus_filtered)

# CLR rows sum to 0 by definition of centered log-ratio
print(f"CLR row sums: max |sum| = {np.abs(X_clr.sum(axis=1)).max():.2e}  (should be ~0)")
print(f"CLR matrix shape: {X_clr.shape}")

## 6. PCA (Aitchison PCA)

PCA on the CLR matrix = Aitchison PCA. The principal components capture the main
directions of compositional variation. We reduce from 303 CLR coordinates to a
small number of PCs for TDA — the scree plot below shows how many components to keep.

In [ ]:
pca = PCA(n_components=N_PCA_COMPONENTS, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_clr)   # shape: (n_samples, N_PCA_COMPONENTS)

evr = pca.explained_variance_ratio_
cumulative_evr = np.cumsum(evr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # left: scree, right: PC1 vs PC2

# Left: scree plot — look for the elbow to choose n_components for TDA
ax = axes[0]
ax.bar(
    range(1, len(evr) + 1), evr * 100,
    color="#4C72B0", alpha=0.7,
    label="per-component",
)
ax2 = ax.twinx()   # cumulative variance on a second y-axis
ax2.plot(
    range(1, len(cumulative_evr) + 1), cumulative_evr * 100,
    color="#DD8452", marker="o", markersize=3, linewidth=1.5,
    label="cumulative",
)
ax2.axhline(80, color="grey", linestyle=":", linewidth=1)  # 80% variance reference line
ax2.set_ylabel("Cumulative variance explained (%)", fontsize=9, color="#DD8452")
ax2.tick_params(axis="y", labelcolor="#DD8452")
ax.set_xlabel("Principal component", fontsize=10)
ax.set_ylabel("Variance explained (%)", fontsize=10)
ax.set_title("Scree plot — Aitchison PCA (CLR)", fontsize=11)
ax.set_xlim(0.5, N_PCA_COMPONENTS + 0.5)

# Right: PC1 vs PC2 colored by sample_type
ax3 = axes[1]
for group, color in GROUP_COLORS.items():
    mask = group_labels == group
    ax3.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=color, label=group,
        s=25,          # point size — increase if hard to see
        alpha=0.7,
        edgecolors="none",
    )
ax3.set_xlabel(f"PC1 ({evr[0]:.1%} variance)", fontsize=10)
ax3.set_ylabel(f"PC2 ({evr[1]:.1%} variance)", fontsize=10)
ax3.set_title("Aitchison PCA — PC1 vs PC2", fontsize=11)
ax3.legend(fontsize=9, loc="best")   # legend position

plt.tight_layout()
save_fig(fig, "07b_cfm_its1_tda_pca")
plt.show()

# Print how many PCs are needed to explain 80% / 90% of variance
for target in (0.80, 0.90):
    n = int(np.searchsorted(cumulative_evr, target)) + 1
    print(f"{target:.0%} variance explained by {n} PCs")

**Choose `N_TDA_COMPONENTS` below based on the scree plot.**
A common rule: take components up to the elbow, or enough for 80% explained variance.
For TDA, fewer dimensions = faster Alpha complex and less noise.

In [ ]:
# Set based on scree plot — change this after inspecting the plot above
N_TDA_COMPONENTS = 119

X_tda = X_pca[:, :N_TDA_COMPONENTS]   # (294, N_TDA_COMPONENTS) — input for Alpha complex

var_captured = cumulative_evr[N_TDA_COMPONENTS - 1]
print(f"TDA input: {X_tda.shape[0]} samples x {N_TDA_COMPONENTS} PCs")
print(f"Variance captured: {var_captured:.1%}")

## 7. Persistent homology — all samples

**Alpha complex vs Rips complex:**  
Alpha complex requires Delaunay triangulation in d dimensions — intractable for d=90.
Instead we use **Vietoris-Rips complex** built from the pairwise distance matrix.
Euclidean distances in 90D CLR-PCA space equal Aitchison distances, so the topology
is identical to what Alpha complex would give in 2D PCA.

**Homology groups:**
- **H0** — connected components. Long-lived bars = isolated clusters of samples.
- **H1** — 1-cycles (loops). Long-lived bars = robust circular structure in the data.

`max_dimension=2` in `create_simplex_tree` expands to triangles (2-simplices),
which is required to detect and kill H1 features.

In [ ]:
from scipy.spatial.distance import pdist, squareform

# Pairwise Euclidean distances in CLR-PCA space = Aitchison distances
dist_matrix = squareform(pdist(X_tda, metric="euclidean"))  # (294, 294)
print(f"Distance matrix: {dist_matrix.shape}")
print(f"Distance range: {dist_matrix[dist_matrix > 0].min():.2f} – {dist_matrix.max():.2f}")

# Build Rips complex from distance matrix
# max_edge_length=inf includes all edges; GUDHI prunes during persistence computation
print("Building Rips complex (all samples) ...")
rips = gudhi.RipsComplex(distance_matrix=dist_matrix)
st_all = rips.create_simplex_tree(max_dimension=2)  # expand to triangles for H1
print(f"Simplices: {st_all.num_simplices():,}")

print("Computing persistence ...")
st_all.compute_persistence()
print("Done.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))  # left: barcode, right: persistence diagram

gudhi.plot_persistence_barcode(
    st_all.persistence(),
    colormap=((0.298, 0.447, 0.690), (0.867, 0.518, 0.322), (0.173, 0.627, 0.173)),   # (r,g,b) tuples per dim: H0=blue, H1=orange
    axes=axes[0],
    inf_delta=0.1,
)
axes[0].set_title("Persistence barcode — all samples (CFM ITS1)", fontsize=11)
axes[0].set_xlabel("Filtration value (Aitchison distance)", fontsize=9)

gudhi.plot_persistence_diagram(
    st_all.persistence(),
    colormap=((0.298, 0.447, 0.690), (0.867, 0.518, 0.322), (0.173, 0.627, 0.173)),
    axes=axes[1],
    inf_delta=0.1,
)
axes[1].set_title("Persistence diagram — all samples (CFM ITS1)", fontsize=11)

plt.tight_layout()
save_fig(fig, "07b_cfm_its1_tda_ph_all")
plt.show()

pers = st_all.persistence()
h0 = [(b, d) for dim, (b, d) in pers if dim == 0 and d != float("inf")]
h1 = [(b, d) for dim, (b, d) in pers if dim == 1]
print(f"H0 finite features: {len(h0)}")
print(f"H1 features:        {len(h1)}")
if h1:
    lt = sorted([d - b for b, d in h1], reverse=True)
    print(f"H1 lifetimes: max={lt[0]:.3f}, top-3={[round(x,3) for x in lt[:3]]}")

## 8. PH per group — bagged vs unbagged

Separate Rips complexes on each group subsample. Compares the topological
structure of pollinator-excluded vs open flowers independently.

In [ ]:
def compute_ph(X: np.ndarray) -> gudhi.SimplexTree:
    """Build Rips complex and compute persistence on a point cloud."""
    dm = squareform(pdist(X, metric="euclidean"))
    st = gudhi.RipsComplex(distance_matrix=dm).create_simplex_tree(max_dimension=2)
    st.compute_persistence()
    return st


bagged_mask   = group_labels == "bagged_flower"
unbagged_mask = group_labels == "unbagged_flower"

print("Computing PH for bagged_flower ...")
st_bagged   = compute_ph(X_tda[bagged_mask])
print("Computing PH for unbagged_flower ...")
st_unbagged = compute_ph(X_tda[unbagged_mask])
print("Done.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))  # rows: bagged / unbagged; cols: barcode / diagram

for row, (st, label) in enumerate([
    (st_bagged,   "bagged_flower"),
    (st_unbagged, "unbagged_flower"),
]):
    gudhi.plot_persistence_barcode(
        st.persistence(),
        colormap=((0.298, 0.447, 0.690), (0.867, 0.518, 0.322), (0.173, 0.627, 0.173)),
        axes=axes[row, 0],
        inf_delta=0.1,
    )
    axes[row, 0].set_title(f"Barcode — {label}", fontsize=10)
    axes[row, 0].set_xlabel("Filtration value", fontsize=9)

    gudhi.plot_persistence_diagram(
        st.persistence(),
        colormap=((0.298, 0.447, 0.690), (0.867, 0.518, 0.322), (0.173, 0.627, 0.173)),
        axes=axes[row, 1],
        inf_delta=0.1,
    )
    axes[row, 1].set_title(f"Persistence diagram — {label}", fontsize=10)

    h1 = [(b, d) for dim, (b, d) in st.persistence() if dim == 1]
    if h1:
        lt = sorted([d - b for b, d in h1], reverse=True)
        print(f"{label}: {len(h1)} H1 features, max lifetime={lt[0]:.3f}")
    else:
        print(f"{label}: 0 H1 features")

plt.tight_layout()
save_fig(fig, "07b_cfm_its1_tda_ph_groups")
plt.show()

## 9. Wasserstein distance + permutation test

Formal test: are the H1 topologies of bagged and unbagged groups significantly different?

**Wasserstein distance** (order 1) between two persistence diagrams measures the cost of
the optimal matching between their H1 features — large distance = diagrams are different.

**Permutation test**: randomly reassign samples to two groups of the same sizes (147/147),
recompute PH and Wasserstein distance. Repeat N times to build a null distribution.
p-value = fraction of permutations with distance ≥ observed.

> ⚠️ Each permutation rebuilds two Rips complexes on 147 points — expect ~5–10 min for 99 permutations.

In [ ]:
import time

def compute_ph_dm(dm: np.ndarray) -> gudhi.SimplexTree:
    """Build Rips complex from a precomputed distance matrix and compute persistence."""
    st = gudhi.RipsComplex(distance_matrix=dm).create_simplex_tree(max_dimension=2)
    st.compute_persistence()
    return st


def get_h1_dgm(st: gudhi.SimplexTree) -> np.ndarray:
    """Extract finite H1 persistence pairs as (n, 2) array of (birth, death)."""
    pairs = [(b, d) for dim, (b, d) in st.persistence()
             if dim == 1 and d != float("inf")]
    return np.array(pairs) if pairs else np.empty((0, 2))


# Observed Wasserstein distance between bagged and unbagged H1 diagrams
dgm_bag   = get_h1_dgm(st_bagged)
dgm_unbag = get_h1_dgm(st_unbagged)
w_obs = gudhi.wasserstein.wasserstein_distance(dgm_bag, dgm_unbag, order=1)
print(f"Observed Wasserstein distance (H1): {w_obs:.4f}")
print(f"  bagged H1 features:   {len(dgm_bag)}")
print(f"  unbagged H1 features: {len(dgm_unbag)}")

In [ ]:
N_PERM   = 999    # increase to 999 on a cluster for a more precise p-value
n_total  = len(group_labels)
n_bag    = int(bagged_mask.sum())

rng_perm  = np.random.default_rng(RANDOM_SEED)
perm_dists = np.empty(N_PERM)

t0 = time.time()
for k in range(N_PERM):
    perm = rng_perm.permutation(n_total)
    idx1, idx2 = perm[:n_bag], perm[n_bag:]

    # Submatrices from the precomputed distance matrix (avoids recomputing pairwise distances)
    dm1 = dist_matrix[np.ix_(idx1, idx1)]
    dm2 = dist_matrix[np.ix_(idx2, idx2)]

    st1 = compute_ph_dm(dm1)
    st2 = compute_ph_dm(dm2)
    perm_dists[k] = gudhi.wasserstein.wasserstein_distance(
        get_h1_dgm(st1), get_h1_dgm(st2), order=1
    )

    if (k + 1) % 10 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (k + 1) * (N_PERM - k - 1)
        print(f"  {k+1}/{N_PERM}  elapsed={elapsed:.0f}s  ETA={eta:.0f}s")

p_value = float((perm_dists >= w_obs).sum() + 1) / (N_PERM + 1)  # +1 to avoid p=0S
print(f"\nWasserstein distance (H1): observed={w_obs:.4f}")
print(f"p-value ({N_PERM} permutations): {p_value:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(
    perm_dists,
    bins=20,
    color="#9ecae1", edgecolor="#3182bd", linewidth=0.5,
    label=f"Null distribution ({N_PERM} permutations)",
)
ax.axvline(
    w_obs,
    color="#d62728", linewidth=2, linestyle="--",
    label=f"Observed = {w_obs:.3f}  (p = {p_value:.3f})",
)
ax.set_xlabel("Wasserstein distance (H1)", fontsize=10)
ax.set_ylabel("Count", fontsize=10)
ax.set_title(
    "Permutation test — H1 topology: bagged vs unbagged (CFM ITS1)",
    fontsize=11,
)
ax.legend(fontsize=9)   # legend position — move with loc= if overlapping bars

plt.tight_layout()
save_fig(fig, "07b_cfm_its1_tda_wasserstein_perm")
plt.show()

## 10. Mapper

Mapper builds a graph summary of the data that can be colored by any variable:
1. **Lens** — 1D filter function (here: PC1 = main direction of compositional variation)
2. **Cover** — overlapping intervals along the lens
3. **Clustering** — DBSCAN within each interval's preimage
4. **Graph** — one node per cluster; edge if two clusters share samples
5. **Coloring** — node color = mean of any metadata variable over its member samples

Node size = number of samples. Node position = mean PC1 / PC2 of member samples.

In [ ]:
import networkx as nx
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import pdist


def run_mapper(X, lens, n_intervals=10, overlap=0.5, min_samples=2, eps_pct=30):
    """
    Minimal Mapper implementation.

    X           : (n, d) point cloud — used for clustering within each interval
    lens        : (n,) 1D filter values
    n_intervals : number of cover elements (more = finer resolution)
    overlap     : fractional overlap between consecutive intervals (0–1)
    min_samples : DBSCAN min_samples parameter
    eps_pct     : percentile of pairwise distances used as DBSCAN epsilon
    """
    l_min, l_max = lens.min(), lens.max()
    width = (l_max - l_min) / (n_intervals - (n_intervals - 1) * overlap)
    step  = width * (1 - overlap)

    nodes = {}   # node_id -> frozenset of sample indices
    node_id = 0

    for i in range(n_intervals):
        lo = l_min + i * step
        hi = lo + width
        idx = np.where((lens >= lo) & (lens <= hi))[0]
        if len(idx) < min_samples:
            continue

        X_bin = X[idx]
        if len(idx) < 3:
            labels = np.zeros(len(idx), dtype=int)
        else:
            # Auto-select epsilon from percentile of within-bin pairwise distances
            eps = np.percentile(pdist(X_bin), eps_pct)
            labels = DBSCAN(eps=max(eps, 1e-6), min_samples=min_samples).fit_predict(X_bin)

        for label in set(labels):
            if label == -1:   # DBSCAN noise
                continue
            members = frozenset(idx[labels == label])
            nodes[node_id] = members
            node_id += 1

    # Build graph: edge if two nodes share at least one sample
    G = nx.Graph()
    G.add_nodes_from(nodes.keys())
    node_items = list(nodes.items())
    for i, (nid1, m1) in enumerate(node_items):
        for nid2, m2 in node_items[i + 1:]:
            if m1 & m2:
                G.add_edge(nid1, nid2)

    return G, nodes


# Lens = PC1 (main axis of compositional variation in Aitchison space)
lens = X_pca[:, 0]

G, nodes = run_mapper(
    X_tda, lens,
    n_intervals=12,   # increase for finer resolution
    overlap=0.5,      # 50% overlap is standard
    min_samples=3,
    eps_pct=30,       # DBSCAN eps = 30th percentile of within-bin distances
)

print(f"Mapper graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
sizes = [len(nodes[n]) for n in G.nodes()]
print(f"Node sizes: min={min(sizes)}, median={int(np.median(sizes))}, max={max(sizes)}")

In [ ]:
# Farm / management ordering (inside_forest → full_sun deforestation gradient)
FARM_ORDER = ["ib", "vr", "sa", "kk", "mt", "vi", "yb"]
MGMT_ORDER = ["inside_forest", "near_forest", "agroforest", "full_sun"]


def prep_color(meta_df, col, ordering=None):
    """
    Encode a metadata column to a float array suitable for draw_mapper.

    ordering : list — explicit category order for integer encoding.
               Entries absent from `ordering` (and NaN) become NaN in output.

    Binary columns (yes/no, True/False, 1/0): encoded as 1.0 / 0.0 / NaN.
    Ordered/nominal categories             : encoded as integer index / NaN.
    Numeric                                : coerced to float / NaN.
    """
    s        = meta_df[col]
    vals     = s.values
    non_null = s.dropna()
    str_set  = {str(v).strip().lower() for v in non_null.unique()}

    # Detect boolean / yes-no columns
    BOOL_MAP = {"yes": 1.0, "no": 0.0, "true": 1.0, "false": 0.0,
                "1": 1.0, "0": 0.0, "1.0": 1.0, "0.0": 0.0}
    if str_set <= BOOL_MAP.keys():
        return np.array(
            [BOOL_MAP.get(str(v).strip().lower(), np.nan) if pd.notna(v) else np.nan
             for v in vals],
            dtype=float,
        )

    # Categorical / string → integer index
    if s.dtype == object or hasattr(s, "cat"):
        if ordering is None:
            ordering = sorted(non_null.unique())
        code_map = {v: float(i) for i, v in enumerate(ordering)}
        return np.array(
            [code_map.get(v, np.nan) if pd.notna(v) else np.nan for v in vals],
            dtype=float,
        )

    # Numeric
    return pd.to_numeric(s, errors="coerce").values.astype(float)


def draw_mapper(G, nodes, color_values, ax,
                title="", cmap="RdYlBu_r", colorbar_label="",
                pos=None, bad_color="lightgrey",
                tick_labels=None, node_level_values=None):
    """
    Draw Mapper graph colored by a pre-encoded float array.

    color_values     : (n_samples,) float array — use prep_color() to encode.
                       NaN = missing data; those nodes are drawn in bad_color.
    pos              : node positions dict; computed once, reused across panels.
    bad_color        : colour for NaN nodes and all-NaN panels.
    tick_labels      : list of strings placed at integer tick positions (0, 1, 2, …)
                       on the colorbar; use for categorically-encoded variables so the
                       colorbar shows category names instead of raw integers.
    node_level_values: dict {node_id: float} — when provided, used directly as node
                       colors instead of computing the mean of color_values over node
                       members. Use for metrics that are inherently node-level
                       (e.g. distinct tree count per node).
    """
    # Node position = mean PC1 (x) and PC2 (y) of member samples
    if pos is None:
        pos = {
            nid: (X_pca[list(members), 0].mean(),
                  X_pca[list(members), 1].mean())
            for nid, members in nodes.items()
            if nid in G.nodes()
        }

    # Node color = NaN-safe mean over member samples
    node_colors = []
    for nid in G.nodes():
        vals  = color_values[list(nodes[nid])]
        valid = vals[~np.isnan(vals)]
        node_colors.append(valid.mean() if len(valid) > 0 else np.nan)
    node_colors = np.array(node_colors, dtype=float)

    # Override with pre-computed node-level values when provided
    if node_level_values is not None:
        node_colors = np.array(
            [node_level_values.get(nid, np.nan) for nid in G.nodes()],
            dtype=float,
        )

    # Node size proportional to member count (scale factor: increase for larger nodes)
    node_sizes = [len(nodes[nid]) * 30 for nid in G.nodes()]

    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.4, edge_color="grey", width=1.0)

    # Guard: all nodes lack data for this variable
    if np.all(np.isnan(node_colors)):
        nx.draw_networkx_nodes(G, pos, ax=ax, node_color=bad_color,
                               node_size=node_sizes, alpha=0.5)
        ax.set_title(title + "\n(no data)", fontsize=9)
        ax.set_xlabel("Mean PC1", fontsize=8)
        ax.set_ylabel("Mean PC2", fontsize=8)
        ax.axis("on")
        return pos

    # Partial NaN: show missing nodes in bad_color via colormap bad colour
    cmap_obj = plt.get_cmap(cmap).copy()
    cmap_obj.set_bad(color=bad_color, alpha=0.5)  # bad_color: colour for NaN nodes
    masked = np.ma.masked_invalid(node_colors)

    sc = nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_color=masked,
        node_size=node_sizes,
        cmap=cmap_obj,
        vmin=float(np.nanmin(node_colors)),
        vmax=float(np.nanmax(node_colors)),
        alpha=0.9,
    )
    cb = plt.colorbar(sc, ax=ax, label=colorbar_label, shrink=0.7)  # shrink: adjust if clipped
    if tick_labels is not None:
        # Replace numeric ticks with category names at integer positions 0, 1, 2, …
        cb.set_ticks(np.arange(len(tick_labels), dtype=float))
        cb.set_ticklabels(tick_labels, fontsize=7)

    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Mean PC1", fontsize=8)
    ax.set_ylabel("Mean PC2", fontsize=8)
    ax.axis("on")
    return pos  # return so same layout is reused across all panels


# Panel definitions: (column, title, colormap, colorbar_label, category_ordering, tick_labels)
# tick_labels: list of category names placed at integer colorbar ticks; None = numeric ticks
PANELS = [
    # Row 1: sample identity
    ("sample_type",
     "sample_type\n(0=bagged_flower, 1=unbagged_flower)",
     "coolwarm", "fraction unbagged_flower", None,
     ["bagged_flower", "unbagged_flower"]),
    ("tree_id",
     "tree_id\n(distinct trees per node)",
     "YlOrRd", "distinct trees in node", None,
     None),
    ("farm_id",
     "farm_id\n(inside_forest→full_sun)",
     "RdYlGn_r", "farm", FARM_ORDER,
     FARM_ORDER),
    # Row 2: management + pollination via DIC
    ("management_type",
     "management_type",
     "RdYlGn", "management type", MGMT_ORDER,
     MGMT_ORDER),
    ("is_pollination_dic",
     "is_pollination_dic\n(1=yes)",
     "coolwarm", "pollinated DIC", None,
     None),
    ("pi_dic",
     "pi_dic\n(pollen index DIC)",
     "YlOrRd", "pollen index DIC", None,
     None),
    # Row 3: pollination via CLSM + germination
    ("is_pollination_clsm",
     "is_pollination_clsm\n(1=yes)",
     "coolwarm", "pollinated CLSM", None,
     None),
    ("pi_clsm",
     "pi_clsm\n(pollen index CLSM)",
     "YlOrRd", "pollen index CLSM", None,
     None),
    ("is_germination",
     "is_germination\n(1=yes)",
     "coolwarm", "germination", None,
     None),
    # Row 4: fungal colonization outcomes
    ("callose_plug_index",
     "callose_plug_index",
     "YlOrRd", "callose plug index", None,
     None),
    ("is_hyphae_present",
     "is_hyphae_present\n(1=yes)",
     "coolwarm", "hyphae present", None,
     None),
    ("fungal_colonization_score",
     "fungal_colonization_score",
     "YlOrRd", "fungal colonization score", None,
     None),
]

fig, axes = plt.subplots(4, 3, figsize=(18, 22))  # figsize: (width, height) in inches
fig.suptitle("ITS1 OTU97 Mapper — coloured by metadata variables", fontsize=13)

# tree_id: pre-extract array for distinct-tree-count computation
tree_ids_arr = meta["tree_id"].values

pos = None  # computed on first panel, reused for all subsequent panels
for idx, (col, title, cmap, cb_label, ordering, tick_labels) in enumerate(PANELS):
    ax = axes[idx // 3, idx % 3]

    if col == "tree_id":
        # Distinct tree count per node — node-level metric; mean of integer-encoded IDs is
        # meaningless for 49 unordered categories
        node_tree_counts = {
            nid: float(len(set(tree_ids_arr[list(members)])))
            for nid, members in nodes.items()
        }
        cv = np.zeros(len(meta))   # placeholder; overridden by node_level_values
        pos = draw_mapper(G, nodes, cv, ax,
                          title=title, cmap=cmap, colorbar_label=cb_label,
                          pos=pos, tick_labels=tick_labels,
                          node_level_values=node_tree_counts)
    else:
        cv  = prep_color(meta, col, ordering=ordering)
        pos = draw_mapper(G, nodes, cv, ax,
                          title=title, cmap=cmap, colorbar_label=cb_label,
                          pos=pos, tick_labels=tick_labels)

fig.tight_layout()
save_fig(fig, "07b_cfm_its1_tda_mapper")
plt.show()


In [ ]:
PDF_ALL.close()
print(f"All figures saved to:  {FIGURES_DIR}")
print(f"Multipage PDF:         {FIGURES_DIR / '07b_cfm_its1_tda_all_figures.pdf'}")